# Millionaire Project - Large Experiments

Use this notebook in Google Colab when your local machine is too slow for repeated seeds, larger backtests, or parameter sweeps.

Run the cells from top to bottom. Start with the small settings first, then scale up.

## 1. Get The Project

This clones the GitHub repository into the Colab runtime. If you opened this notebook directly from GitHub, you still need this step because Colab loads the notebook file, not the whole repo.

In [1]:
!git clone https://github.com/charlesclothilde-cmd/MillionaireProject.git
%cd MillionaireProject
!pip install -q -r requirements.txt

Cloning into 'MillionaireProject'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 48 (delta 15), reused 45 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 68.45 KiB | 17.11 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/MillionaireProject
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 69.1 MB/s eta 0:00:00


Optional: mount Google Drive if you want outputs to survive after the Colab session ends.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load The Data And Helpers

In [3]:
from pathlib import Path
import pandas as pd

from lottery_data import (
    load_draws,
    data_freshness_label,
    run_repeated_backtests,
    summarise_repeated_backtests,
    run_parameter_sweep,
    backtest_model_ablation,
    summarise_backtest,
    summarise_prize_tier_values,
)

df = load_draws('euromillions_full_2004_2026.csv')
print(data_freshness_label(df))
print(f'Draws loaded: {len(df):,}')

output_dir = Path('/content/drive/MyDrive/MillionaireProject_outputs')
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Outputs will be saved to: {output_dir}')

Dataset updated through 2026-06-26
Draws loaded: 1,958
Outputs will be saved to: /content/drive/MyDrive/MillionaireProject_outputs


## 3. Smoke Test

Run this first. If this works, the full project is available in Colab.

In [4]:
smoke = run_repeated_backtests(
    df,
    seeds=list(range(3)),
    training_window=300,
    test_draws=25,
    recent_draw_count=100,
    simulations=50,
    half_life=75,
)

smoke_summary = summarise_repeated_backtests(smoke)
smoke_summary

,Strategy,runs,mean_total_matches,total_ci_low,total_ci_high,mean_prize_hit_rate,hit_ci_low,hit_ci_high,mean_expected_prize_value,value_ci_low,value_ci_high,mean_expected_net_value,mean_roi,win_rate_vs_random,avg_total_edge_vs_random,avg_hit_edge_vs_random,avg_value_edge_vs_random
0,Random baseline,3,0.920000,0.684804,1.155196,0.093333,0.067200,0.119466,0.440000,0.242701,0.637299,-2.060000,-0.824000,NaN,NaN,NaN,NaN
1,Recent hot,3,0.866667,0.609288,1.124045,0.080000,0.001601,0.158399,0.426667,-0.048779,0.902112,-2.073333,-0.829333,0.333333,-0.053333,-0.013333,-0.013333
2,Ranked model,3,0.880000,0.553601,1.206399,0.080000,-0.039756,0.199756,0.386667,-0.257178,1.030511,-2.113333,-0.845333,0.333333,-0.040000,-0.013333,-0.053333
3,Balanced mix,3,0.760000,0.603203,0.916797,0.066667,0.014401,0.118932,0.306667,0.102563,0.510771,-2.193333,-0.877333,0.333333,-0.160000,-0.026667,-0.133333
4,Cold numbers,3,0.706667,0.535302,0.878031,0.080000,0.034737,0.125263,0.266667,0.083737,0.449597,-2.233333,-0.893333,0.000000,-0.213333,-0.013333,-0.173333


## 4. Recommended Large Run

This is a good first serious Colab experiment: more seeds before very large simulation counts.

In [5]:
repeated = run_repeated_backtests(
    df,
    seeds=list(range(100)),
    training_window=300,
    test_draws=200,
    recent_draw_count=100,
    simulations=100,
    half_life=250,
)

repeated_summary = summarise_repeated_backtests(repeated)

repeated.to_csv(output_dir / 'large_repeated_runs_100_seeds.csv', index=False)
repeated_summary.to_csv(output_dir / 'large_repeated_summary_100_seeds.csv', index=False)

repeated_summary

,Strategy,runs,mean_total_matches,total_ci_low,total_ci_high,mean_prize_hit_rate,hit_ci_low,hit_ci_high,mean_expected_prize_value,value_ci_low,value_ci_high,mean_expected_net_value,mean_roi,win_rate_vs_random,avg_total_edge_vs_random,avg_hit_edge_vs_random,avg_value_edge_vs_random
0,Cold numbers,100,0.80065,0.791716,0.809584,0.07735,0.074044,0.080656,8.77085,-7.791730,25.333430,6.27085,2.50834,0.33,-0.02705,0.0030,8.44605
1,Balanced mix,100,0.82235,0.810900,0.833800,0.07875,0.075088,0.082412,0.33235,0.314376,0.350324,-2.16765,-0.86706,0.49,-0.00535,0.0044,0.00755
2,Ranked model,100,0.83485,0.822506,0.847194,0.07815,0.074950,0.081350,0.33055,0.314363,0.346737,-2.16945,-0.86778,0.54,0.00715,0.0038,0.00575
3,Random baseline,100,0.82770,0.816203,0.839197,0.07435,0.070835,0.077865,0.32480,0.306430,0.343170,-2.17520,-0.87008,NaN,NaN,NaN,NaN
4,Recent hot,100,0.81735,0.808394,0.826306,0.07415,0.070428,0.077872,0.31950,0.299367,0.339633,-2.18050,-0.87220,0.46,-0.01035,-0.0002,-0.00530


In [8]:
repeated[repeated["Strategy"] == "Cold numbers"].sort_values(
    "expected_prize_value",
    ascending=False
).head(10)

,seed,Strategy,draws,avg_ball_matches,avg_star_matches,avg_total_matches,total_ci_low,total_ci_high,prize_hits,prize_hit_rate,hit_ci_low,hit_ci_high,expected_prize_value,expected_net_value,return_ratio,roi,total_prize_value,best_prize_tier,best_prize_rank,best_prize_value
450,90,Cold numbers,200,0.515,0.335,0.850,0.723612,0.976388,15,0.075,0.045975,0.120044,845.365,842.865,338.146,337.146,169073.0,5+1,2,169001.0
30,6,Cold numbers,200,0.540,0.330,0.870,0.745489,0.994511,19,0.095,0.061663,0.143602,0.640,-1.860,0.256,-0.744,128.0,3+2,6,48.0
215,43,Cold numbers,200,0.540,0.345,0.885,0.759819,1.010181,20,0.100,0.065670,0.149406,0.610,-1.890,0.244,-0.756,122.0,3+2,6,48.0
115,23,Cold numbers,200,0.470,0.355,0.825,0.702736,0.947264,15,0.075,0.045975,0.120044,0.595,-1.905,0.238,-0.762,119.0,3+2,6,48.0
180,36,Cold numbers,200,0.485,0.280,0.765,0.650448,0.879552,18,0.090,0.057687,0.137766,0.570,-1.930,0.228,-0.772,114.0,3+2,6,48.0
10,2,Cold numbers,200,0.505,0.315,0.820,0.699869,0.940131,17,0.085,0.053746,0.131896,0.505,-1.995,0.202,-0.798,101.0,4+0,7,33.0
481,96,Cold numbers,200,0.545,0.265,0.810,0.690975,0.929025,23,0.115,0.077864,0.166647,0.455,-2.045,0.182,-0.818,91.0,3+1,9,9.0
121,24,Cold numbers,200,0.545,0.340,0.885,0.761370,1.008630,20,0.100,0.065670,0.149406,0.455,-2.045,0.182,-0.818,91.0,2+2,8,12.0
161,32,Cold numbers,200,0.540,0.350,0.890,0.772279,1.007721,19,0.095,0.061663,0.143602,0.445,-2.055,0.178,-0.822,89.0,2+2,8,12.0
455,91,Cold numbers,200,0.530,0.335,0.865,0.749024,0.980976,21,0.105,0.069707,0.155180,0.445,-2.055,0.178,-0.822,89.0,3+0,10,8.0


In [9]:
repeated.groupby("Strategy")[[
    "avg_total_matches",
    "prize_hit_rate",
    "expected_prize_value",
    "roi"
]].median().sort_values("expected_prize_value", ascending=False)

,avg_total_matches,prize_hit_rate,expected_prize_value,roi
Strategy,,,,
Ranked model,0.830,0.075,0.3275,-0.869
Balanced mix,0.825,0.075,0.3175,-0.873
Random baseline,0.830,0.075,0.3150,-0.874
Cold numbers,0.805,0.080,0.3125,-0.875
Recent hot,0.815,0.075,0.3100,-0.876


## 5. Parameter Sweep

Keep this bounded at first. Parameter sweeps multiply quickly.

In [6]:
sweep = run_parameter_sweep(
    df,
    half_lives=[75, 150, 250, 400],
    simulation_counts=[100, 300],
    training_windows=[300, 500],
    recent_draw_counts=[50, 100],
    test_draws=200,
    random_seed=42,
)

sweep.to_csv(output_dir / 'large_parameter_sweep.csv', index=False)
sweep.head(20)

,half_life,simulations,training_window,test_draws,recent_draw_count,avg_total_matches,total_ci_low,total_ci_high,prize_hit_rate,hit_ci_low,...,expected_prize_value,expected_net_value,return_ratio,roi,match_0_rate,match_1_rate,match_2_rate,match_3_plus_rate,total_match_std,best_prize_tier
4,250,100,300,200,50,0.955,0.829726,1.080274,0.135,0.094466,...,0.620,-1.880,0.248,-0.752,0.345,0.425,0.175,0.055,0.901651,2+2
12,250,100,300,200,100,0.955,0.829726,1.080274,0.135,0.094466,...,0.620,-1.880,0.248,-0.752,0.345,0.425,0.175,0.055,0.901651,2+2
6,400,100,300,200,50,0.965,0.839665,1.090335,0.130,0.090282,...,0.610,-1.890,0.244,-0.756,0.340,0.425,0.180,0.055,0.902095,2+2
14,400,100,300,200,100,0.965,0.839665,1.090335,0.130,0.090282,...,0.610,-1.890,0.244,-0.756,0.340,0.425,0.180,0.055,0.902095,2+2
2,150,100,300,200,50,0.905,0.781824,1.028176,0.120,0.081980,...,0.565,-1.935,0.226,-0.774,0.365,0.430,0.155,0.050,0.886552,2+2
10,150,100,300,200,100,0.905,0.781824,1.028176,0.120,0.081980,...,0.565,-1.935,0.226,-0.774,0.365,0.430,0.155,0.050,0.886552,2+2
0,75,100,300,200,50,0.905,0.784198,1.025802,0.115,0.077864,...,0.505,-1.995,0.202,-0.798,0.365,0.425,0.155,0.055,0.869468,2+2
8,75,100,300,200,100,0.905,0.784198,1.025802,0.115,0.077864,...,0.505,-1.995,0.202,-0.798,0.365,0.425,0.155,0.055,0.869468,2+2
16,75,100,500,200,50,0.890,0.769845,1.010155,0.105,0.069707,...,0.475,-2.025,0.190,-0.810,0.370,0.430,0.145,0.055,0.864812,2+2
24,75,100,500,200,100,0.890,0.769845,1.010155,0.105,0.069707,...,0.475,-2.025,0.190,-0.810,0.370,0.430,0.145,0.055,0.864812,2+2


## 6. Ablation Retest

This checks whether the no-spread result still looks interesting over a larger window.

In [7]:
ablation = backtest_model_ablation(
    df,
    training_window=300,
    history_window=300,
    test_draws=200,
    simulations=300,
    half_life=250,
    random_seed=42,
)

ablation_summary = summarise_backtest(ablation)
ablation_tier_values = summarise_prize_tier_values(ablation)

ablation.to_csv(output_dir / 'large_ablation_runs.csv', index=False)
ablation_summary.to_csv(output_dir / 'large_ablation_summary.csv', index=False)
ablation_tier_values.to_csv(output_dir / 'large_ablation_tier_values.csv', index=False)

ablation_summary

,Strategy,draws,avg_ball_matches,avg_star_matches,avg_total_matches,total_ci_low,total_ci_high,prize_hits,prize_hit_rate,hit_ci_low,hit_ci_high,expected_prize_value,expected_net_value,return_ratio,roi,total_prize_value,best_prize_tier,best_prize_rank,best_prize_value
0,No balance score,200,0.490,0.300,0.790,0.677477,0.902523,22,0.110,0.073772,0.160927,0.410,-2.090,0.164,-0.836,82.0,1+2,11,6.0
1,Full model,200,0.500,0.320,0.820,0.708191,0.931809,19,0.095,0.061663,0.143602,0.395,-2.105,0.158,-0.842,79.0,2+2,8,12.0
2,No cold score,200,0.505,0.310,0.815,0.705532,0.924468,19,0.095,0.061663,0.143602,0.385,-2.115,0.154,-0.846,77.0,2+2,8,12.0
3,Only pair score,200,0.510,0.300,0.810,0.694264,0.925736,18,0.090,0.057687,0.137766,0.380,-2.120,0.152,-0.848,76.0,3+0,10,8.0
4,Only balance/spread,200,0.480,0.350,0.830,0.709588,0.950412,10,0.050,0.027383,0.089578,0.315,-2.185,0.126,-0.874,63.0,2+2,8,12.0
5,Only hot score,200,0.480,0.305,0.785,0.675708,0.894292,16,0.080,0.049841,0.125989,0.290,-2.210,0.116,-0.884,58.0,2+1,12,5.0
6,No pair score,200,0.525,0.375,0.900,0.792379,1.007621,14,0.070,0.042152,0.114055,0.285,-2.215,0.114,-0.886,57.0,2+2,8,12.0
7,No spread score,200,0.475,0.300,0.775,0.671552,0.878448,14,0.070,0.042152,0.114055,0.240,-2.260,0.096,-0.904,48.0,2+1,12,5.0
8,No hot score,200,0.450,0.350,0.800,0.698851,0.901149,7,0.035,0.017056,0.070471,0.145,-2.355,0.058,-0.942,29.0,1+2,11,6.0


## Scaling Notes

- Increase seeds before increasing simulations.
- Save CSVs after every long cell.
- If Colab disconnects, rerun setup cells and continue from the saved CSVs in Google Drive.
- Good next steps: compare `half_life=250` vs `400`, and retest the no-spread ablation over more than one seed.